this is the file for running MLs on the given set of multi stock data
- the ML flow:
    - data cleaning
    - insert feature
    - model training:
        - Ridge (mannual tuning)
        - LightGBM (optuna tuning)
        - MLP (optuna tuning, seed stability check)
        - LSTM (optuna tuning, seed stability check)
    - final run on test set

In [ ]:
import numpy as np
import pandas as pd

# import from pipeline
from pipeline.data import load_raw_data, clean_raw_data, drop_known_bad_date, check_monotonic_dates, adjust_units, export_for_qlib
from pipeline.features import build_alpha158, split_X_y_alpah158, build_df_for_LSTM, split_X_y_LSTM
from pipeline.partition import fixed_date_split, get_walk_forward_splits, get_fold_data, build_sequences
from pipeline.evaluate import zero_baseline_metrics, walk_forward_evaluate, compare_models
from pipeline.models import tune_lightgbm, tune_lstm, tune_mlp, tune_ridge_alpha, seed_stability_lstm, seed_stability_mlp, implement_ridge, implement_lgb, implement_lstm,implement_mlp
from pipeline.visual import plot_actual_vs_predicted

# data loading and cleaning

In [ ]:
df = load_raw_data()
df = clean_raw_data(df)
df = drop_known_bad_date(df)
print('dates sorted:', check_monotonic_dates(df) == {})

df = adjust_units(df)
print(df.describe())

export_for_qlib(df)

# building features and splitting X, y
- Ridge, LightGBM and MLP (Qlib Alpha158)
- LSTM (manually)

In [ ]:
# Alpha158 from Qlib
df_qlib = build_alpha158(
    provider_uri="~/.qlib/qlib_data/my_custom_data",
    start_time="2018-01-02",
    end_time="2026-06-05",
    fit_start_time="2018-01-02",
    fit_end_time="2023-12-29",
    instruments=['000016.SH', '000300.SH', '000852.SH', '000905.SH', '000985.CSI',
                 '399303.SZ', '868008.WI', '8841425.WI', '932000.CSI'],)

X_qlib, y_qlib = split_X_y_alpah158(df_qlib)

In [ ]:
# Manual alpha for LSTM
df_lstm = build_df_for_LSTM(df)
X_lstm, y_lstm = split_X_y_LSTM(
    raw_df=df_lstm, 
    feature_cols=["log_ret", "vol_z", "hl_range",
                     "STD20", "RESI30", "IMIN20"])

# data splitting for train, validate and test
- fixed split for seed stability validationa and final check
- walk-forward splits
- sliding window for LSTM training

In [ ]:
X_train, y_train, X_valid, y_valid, X_test, y_test = fixed_date_split(
    X=X_qlib, y=y_qlib, 
    train_dates=["2018-01-02", "2023-12-29"],
    valid_dates=["2024-01-02", "2025-03-14"],
    test_dates=["2025-03-17", "2026-06-05"],)


In [ ]:
all_dates = X_qlib.index.get_level_values('datetime').unique()
splits = get_walk_forward_splits(
    dates=all_dates, 
    initial_train_years=3, 
    valid_months=6, 
    test_months=6, 
    step_months=6)

# get_fold_data to get the actual value

In [ ]:
split = ("2023-12-29", "2025-03-14", "2026-06-05")
X_train_l, y_train_l, X_valid_l, y_valid_l, X_test_l, y_test_l = get_fold_data(X_lstm, y_lstm, split)

X_train_seq, y_train_seq, dates_train_seq, codes_train_seq = build_sequences(X_train_l, y_train_l, seq_len = 20)
X_valid_seq, y_valid_seq, dates_valid_seq, codes_valid_seq = build_sequences(X_valid_l, y_valid_l, seq_len = 20)
X_test_seq, y_test_seq, dates_test_seq, codes_test_seq = build_sequences(X_test_l, y_test_l, seq_len = 20)

# Ridge

In [ ]:
best_alpha = tune_ridge_alpha(X_train, y_train, X_valid, y_valid)

# LightGBM

In [ ]:
best_params_lgb = tune_lightgbm(X_train, y_train, X_valid, y_valid)

# MLP

In [ ]:
best_params_mlp = tune_mlp(X_train, y_train, X_valid, y_valid)

In [ ]:
seeds = [0, 1, 41, 123, 7]
rank_ics = []

rank_ics = seed_stability_mlp(X_train, y_train, X_valid, y_valid, best_params_mlp, seeds)
print(rank_ics)

In [ ]:
# LSTM

In [ ]:
best_params_lstm = tune_lstm(X_train_seq, y_train_seq, X_valid_seq, y_valid_seq, 
                             dates_valid_seq, codes_valid_seq)


In [ ]:
seeds = [0, 1, 41, 123, 7]
rank_ics = []

rank_ics = seed_stability_lstm(X_train_seq, y_train_seq, X_valid_seq, y_valid_seq,
                                dates_valid_seq, codes_valid_seq, seeds, best_params_lstm)
print(rank_ics)

# walk-forward evaluation on full set

In [ ]:
zero_basline = zero_baseline_metrics(y_test)

result_ridge = walk_forward_evaluate(
    implement_ridge,
    X_qlib,
    y_qlib,
    splits,
    best_params=best_alpha,
    combine_train_valid = True, 
) 

result_lgb = walk_forward_evaluate(
    implement_lgb,
    X_qlib,
    y_qlib,
    splits,
    best_params=best_params_lgb,
    combine_train_valid = False, 
) 

result_mlp = walk_forward_evaluate(
    implement_mlp,
    X_qlib,
    y_qlib,
    splits,
    best_params=best_params_mlp,
    combine_train_valid = True, 
) 

result_lstm = walk_forward_evaluate(
    implement_lstm,
    X_lstm,
    y_lstm,
    splits,
    best_params=best_params_mlp,
    combine_train_valid = False, 
) 

combined_fold_results = {'zero basline': zero_basline,
                         'ridge': result_ridge, 
                         'LightGBM': result_lgb,
                         'MLP': result_mlp,
                         'LSTM': result_lstm}

summary = compare_models(combined_fold_results)
print(summary)
# result, walk_forward_evaluate, implement_lstm, evaluate function

# visualisation on one-time test set 

In [ ]:
X_fit = pd.concat([X_train, X_valid], axis=0)
y_fit = pd.concat([y_train, y_valid], axis=0)

preds_ridge, metrix_ridge = implement_ridge(X_fit, y_fit, X_test, y_test, best_alpha)
preds_lgb, metrix_lgb = implement_lgb(X_train, y_train, X_valid, y_valid, X_test, y_test, best_params_lgb)
preds_mlp, metrix_mlp = implement_mlp(X_fit, y_fit, X_test, y_test, best_params_mlp)
preds_lstm, metrix_lstm = implement_lstm(X_train_seq, y_train_seq, X_valid_seq, y_valid_seq, X_test_seq, y_test_seq, best_params_lstm)

plot_actual_vs_predicted(y_test, preds_ridge)
plot_actual_vs_predicted(y_test, preds_lgb)
plot_actual_vs_predicted(y_test, preds_mlp)
plot_actual_vs_predicted(y_test_seq, preds_lstm)